# 春第 5 回｜測定とデータの来歴

学校統計の四つの列を確かめ，児童に配る教材の冊数と，学校ごとの説明件数を見積もります。

最初に本文の例を一緒に実行し，後半では自分で条件を選んで約 20 分の実習に取り組みます。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。 [SSDSE の公式解説 PDF](https://www.nstac.go.jp/files/kaisetsu-A-2026.pdf) は別に開いて参照します。

以下の読み込みセルはそのまま実行します。出力は `1741` です。

In [ ]:
from pathlib import Path
import pandas as pd

file_name = "ssdse-school-2024.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

schools = pd.read_csv(csv_path)
len(schools)

## 本文と一緒に実行

問いを読んでから，コードを上から順に実行します。「提供した作図セル」はそのまま実行し，軸や点，棒の位置を読みます。

### 教材の冊数と学校への説明件数

歌志内市と三島村で，児童に 1 人 1 冊，学校に 1 校 1 件の説明を準備します。二つの図で，同じ地域を比べても必要な数が違う理由を考えます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt
import pandas as pd

opening_schools = schools
opening_match = (opening_schools["code"] == "R01227") | (opening_schools["code"] == "R46303")
opening_regions = opening_schools.loc[opening_match]
opening_pupils = opening_regions["elementary_pupils"] + opening_regions["integrated_early_pupils"]
opening_counts = opening_regions["elementary_schools"] + opening_regions["integrated_schools"]

opening_figure, (pupil_axis, school_axis) = plt.subplots(
    1, 2, sharey=True, layout="constrained"
)
region_labels = ["Utashinai", "Mishima"]
pupil_bars = pupil_axis.barh(region_labels, opening_pupils, color="#286b70")
school_bars = school_axis.barh(region_labels, opening_counts, color="#825039")
pupil_axis.bar_label(pupil_bars, padding=3)
school_axis.bar_label(school_bars, padding=3)
pupil_axis.set(xlim=(0, 42), xticks=[0, 10, 20, 30, 40], xlabel="Pupils")
school_axis.set(xlim=(0, 5), xticks=[0, 1, 2, 3, 4], xlabel="Schools")
pupil_axis.invert_yaxis()
pupil_axis.spines[["top", "right"]].set_visible(False)
school_axis.spines[["top", "right"]].set_visible(False)
plt.show()

### 市区町村の行と元の項目を確認する

この表の一行は一つの市区町村です。先頭の三行で，地域の識別コードと名前を確認します。

In [ ]:
import pandas as pd

lecture_schools = schools.copy()
region_columns = ["code", "prefecture", "municipality"]
print(len(schools))
schools[region_columns].head(3)

歌志内市の行を見ます。公式解説 PDF の項目コードを手掛かりに，四つの列が数える対象と単位を確かめます。

| 項目コード | 数えるもの | 単位 |
| --- | --- | --- |
| E2101 | 小学校 | 校 |
| E2501 | 小学校の児童 | 人 |
| E3901 | 義務教育学校 | 校 |
| E3905 | 義務教育学校前期課程の児童 | 人 |

四つの値は 2024 年 5 月 1 日の学校基本調査に基づきます。

In [ ]:
school_columns = [
    "code", "prefecture", "municipality",
    "elementary_schools", "elementary_pupils",
    "integrated_schools", "integrated_early_pupils",
]
schools.loc[schools["code"] == "R01227", school_columns]

### 同じ二地域の児童と学校をそれぞれ足す

児童数では小学校の児童と義務教育学校前期課程の児童を，学校数では小学校と義務教育学校を足します。歌志内市と三島村について，冒頭の二枚の図と照らし合わせます。

In [ ]:
lecture_schools["early_pupils"] = lecture_schools["elementary_pupils"] + lecture_schools["integrated_early_pupils"]
comparison_regions = (lecture_schools["code"] == "R01227") | (lecture_schools["code"] == "R46303")
pupil_columns = ["municipality", "elementary_pupils", "integrated_early_pupils", "early_pupils"]
lecture_schools.loc[comparison_regions, pupil_columns]

In [ ]:
lecture_schools["school_count"] = lecture_schools["elementary_schools"] + lecture_schools["integrated_schools"]
same_pupils = lecture_schools.loc[comparison_regions]
comparison_columns = ["municipality", "early_pupils", "elementary_schools",
                      "integrated_schools", "school_count"]
same_pupils[comparison_columns]

### 学校あたりの平均と学校ごとの人数

白糠町の 203 人が通う学校は，どの列を足して何校になりますか。同じ対象の学校数で児童数を割ります。

In [ ]:
shiranuka = lecture_schools.loc[lecture_schools["code"] == "R01668"]
shiranuka_columns = [
    "municipality", "elementary_schools", "elementary_pupils",
    "integrated_schools", "integrated_early_pupils", "early_pupils", "school_count",
]
shiranuka[shiranuka_columns]

In [ ]:
shiranuka_pupils_per_school = shiranuka["early_pupils"] / shiranuka["school_count"]
shiranuka_pupils_per_school

二つの図は，3 校の人数の組合せを考えるための仮の例です。合計と学校あたりの平均が同じでも，学校ごとに何冊用意するかがどう変わるか見比べます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

allocation_figure, (first_axis, second_axis) = plt.subplots(
    1, 2, sharex=True, sharey=True, layout="constrained"
)
school_labels = ["Elementary", "Integrated A", "Integrated B"]
school_colors = ["#825039", "#286b70", "#286b70"]
first_bars = first_axis.barh(school_labels, [8, 97, 98], color=school_colors)
second_bars = second_axis.barh(school_labels, [8, 15, 180], color=school_colors)
first_axis.bar_label(first_bars, padding=3)
second_axis.bar_label(second_bars, padding=3)
first_axis.set(title="Hypothetical allocation 1", xlim=(0, 204),
               xticks=[0, 50, 100, 150, 200], xlabel="Pupils")
second_axis.set(title="Hypothetical allocation 2", xlabel="Pupils")
first_axis.invert_yaxis()
first_axis.spines[["top", "right"]].set_visible(False)
second_axis.spines[["top", "right"]].set_visible(False)
plt.show()

### 県内の数量を足す

岐阜県と奈良県の自治体数を確かめます。次に，県ごとの児童数と学校数を別々に足します。自治体の行数，児童の人数，学校の校数を読み分けます。

In [ ]:
prefecture_groups = lecture_schools.groupby("prefecture")
prefecture_names = ["岐阜県", "奈良県"]
prefecture_rows = prefecture_groups.size()
prefecture_rows.loc[prefecture_names]

In [ ]:
prefecture_totals = prefecture_groups[["early_pupils", "school_count"]].sum()
prefecture_totals.loc[prefecture_names]

学校種ごとの内訳に戻ります。県の合計で知りたいことと，一校ごとに用意する数を決めるのに必要な情報を分けて考えます。

In [ ]:
type_columns = ["elementary_pupils", "integrated_early_pupils",
                "elementary_schools", "integrated_schools"]
prefecture_by_type = prefecture_groups[type_columns].sum()
prefecture_by_type.loc[prefecture_names]

## Jupyter 実習（約 20 分）

同じ地域のデータでも，何を準備するかによって必要な列と単位が変わります。北海道か鹿児島県を選び，教材の冊数または学校への説明件数を見積もります。

これは **2024 年 5 月 1 日の記録を使う練習**です。現在の発注・訪問計画には，最新の人数や連絡先などの確認が必要です。共通課題は約 20 分（5・9・6 分），末尾は任意の発展です。

### 1. 地域と依頼を選ぶ（5 分）

地域を **北海道／鹿児島県**から選び，次の依頼を一つ選びます。

- **A 教材**：地域内の小学校の児童と義務教育学校の前期課程の児童へ，1 人に 1 冊を用意する。
- **B 説明**：地域内の小学校と義務教育学校を対象に，学校ごとに 1 件の説明の機会を用意する。

B は学校単位で数える練習です。学校数を建物数や移動回数と同一視しないでください。対象には，特別支援学校などの別の学校種は含めません。

`prefecture` の引用符の中を変更して実行し，その道県の自治体数を確認します。

In [ ]:
prefecture = "北海道"
area = schools.loc[schools["prefecture"] == prefecture]
len(area)

選んだ依頼と，答えの単位を書いてください。

- 道県：
- 依頼 A／B：
- 求める数量の単位と理由：

### 2. 列を選んで計算する（9 分）

以下は選んだ地域の元の 4 列の合計です。`len(area)` の出力は自治体数であり，児童数や学校数とは異なります。

| 列 | 収録内容 | 原項目 |
|---|---|---|
| `elementary_pupils` | 小学校児童数（人） | E2501 |
| `integrated_early_pupils` | 義務教育学校前期課程児童数（人） | E3905 |
| `elementary_schools` | 小学校数（校） | E2101 |
| `integrated_schools` | 義務教育学校数（校） | E3901 |

公式解説の 3 ページで学校種の説明，7 ページで項目と年次を確認します。これらは学校所在地による集計です。

In [ ]:
area[["elementary_pupils", "integrated_early_pupils",
      "elementary_schools", "integrated_schools"]].sum()

依頼に必要な 2 列を足す一行を，次のセルへ書いてください。

```python
quantity = area["一つ目の列名"] + area["二つ目の列名"]
```

`quantity` の一つの値は，1 自治体分の数量です。続くセルではその値をすべて足して道県の合計を求めます。

<details><summary>列の選び方のヒント</summary>

A は児童数の 2 列，B は学校数の 2 列を足します。列の名前が似ていても，人と校は別の単位です。元の列を上書きする必要はありません。

</details>

In [ ]:
# ここに quantity を作る一行を書きます。

上の一行を実行してから，次の 2 行の先頭の `#` を外して実行します。未記入版を上から実行してエラーが出ないことと，課題が完成したことは別です。

In [ ]:
# total = quantity.sum()
# total

### 3. 数量の意味を報告する（6 分）

次の欄に，値だけでなく，その値で何を準備できるかを書いてください。結果の報告には常体を使って構いません。

- **見積り**：どの道県の，どの学校種の，いつの記録を使い，何をいくつ用意する計算になったか。
- **列の選択**：選ばなかった依頼なら，どの 2 列を使い，答えの単位はどう変わるか。
- **次に必要な表**：実際に学校別に配送する，または説明の予定を組むなら，追加で必要な情報を一つ挙げ，理由を書く。

「この地域に住む子ども全員」と「この地域の対象学校に在籍する児童」は同じ対象とは限りません。元データで測っている範囲に沿って報告してください。

**提出前**にカーネルを再起動して最初から順に実行し，計算結果と記入した説明が残ったノートを保存します。任意課題を空欄にしても構いません。

### 発展（任意）

選んだ道県の合計が同じでも，学校別の人数の分布は一つに決まりません。教材を学校別に箱詰めする場面を考え，次に集める表の「1 行」と「必要な列」を提案してください。個人の名簿が必要かどうかも，目的に照らして考えます。

別の道県やもう一方の依頼で計算しても構いません。その場合は地域の選択から再実行し，報告の対象と単位も更新してください。

### 出典

統計センター [SSDSE-A-2026](https://www.nstac.go.jp/use/literacy/ssdse/) を加工。学校の値は 2024 年，地域区分は 2025 年 3 月 31 日時点。2026 は版の名前です。原統計と項目定義は同梱の公式解説，教材の列との対応は `ssdse-sources.md` にあります。